*Import Libraries*

In [1]:
import os
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
os.makedirs("charts", exist_ok=True)
pd.set_option("display.width", 120)


Matplotlib is building the font cache; this may take a moment.


STEP 1: Load And Profile

In [2]:
print("=" * 70)
print("STEP 1: Load and profile the Titanic dataset")
print("=" * 70)

df = sns.load_dataset("titanic")

print("\n--- df.info() ---")
df.info()

STEP 1: Load and profile the Titanic dataset

--- df.info() ---
<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    str     
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    str     
 8   class        891 non-null    category
 9   who          891 non-null    str     
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    str     
 13  alive        891 non-null    str     
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), str(5)
memory usage: 80.7 KB


In [3]:
print("\n--- df.describe() ---")
print(df.describe())

print("\n--- df.shape ---")
print(df.shape)


--- df.describe() ---
         survived      pclass         age       sibsp       parch        fare
count  891.000000  891.000000  714.000000  891.000000  891.000000  891.000000
mean     0.383838    2.308642   29.699118    0.523008    0.381594   32.204208
std      0.486592    0.836071   14.526497    1.102743    0.806057   49.693429
min      0.000000    1.000000    0.420000    0.000000    0.000000    0.000000
25%      0.000000    2.000000   20.125000    0.000000    0.000000    7.910400
50%      0.000000    3.000000   28.000000    0.000000    0.000000   14.454200
75%      1.000000    3.000000   38.000000    1.000000    0.000000   31.000000
max      1.000000    3.000000   80.000000    8.000000    6.000000  512.329200

--- df.shape ---
(891, 15)


In [4]:
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_pct = missing_pct[missing_pct > 0].sort_values(ascending=False)
print("\n--- Missing value % per column (only columns with missing values) ---")
print(missing_pct)

# This is the ONE and ONLY raw load of the dataset. Save it immediately, before
# any cleaning, as the committed offline fallback -- 02_modeling.py reads this
# CSV instead of calling sns.load_dataset again.
df.to_csv("titanic.csv", index=False)
print("\nSaved raw dataset -> titanic.csv (offline fallback, loadable via pd.read_csv).")



--- Missing value % per column (only columns with missing values) ---
deck           77.22
age            19.87
embarked        0.22
embark_town     0.22
dtype: float64

Saved raw dataset -> titanic.csv (offline fallback, loadable via pd.read_csv).


STEP 2: Missing-Value Handling

In [5]:
print("\n" + "=" * 70)
print("STEP 2: Missing-value handling (percentage-threshold rule)")
print("=" * 70)
print("Rule: <5% missing -> drop rows | 5-30% missing -> impute | >30% -> drop column")

df_clean = df.copy()

for col in missing_pct.index:
    pct = missing_pct[col]
    print(f"\n{col}: {pct}% missing")

    if pct < 5:
        before = len(df_clean)
        df_clean = df_clean.dropna(subset=[col])
        print(f"  -> Under 5%: dropping rows with missing {col} "
              f"({before - len(df_clean)} row(s) dropped).")

    elif pct <= 30:
        if pd.api.types.is_numeric_dtype(df_clean[col]):
            fill_val = df_clean[col].median()
            df_clean[col] = df_clean[col].fillna(fill_val)
            print(f"  -> 5-30%: imputing with median ({fill_val:.2f}).")
        else:
            fill_val = df_clean[col].mode()[0]
            df_clean[col] = df_clean[col].fillna(fill_val)
            print(f"  -> 5-30%: imputing with mode ('{fill_val}').")

    else:
        print(f"  -> Over 30% missing: too unreliable to impute confidently.")
        print(f"     Decision: DROP the '{col}' column entirely.")
        print(f"     Justification: at {pct}% missing, more than 3 out of 4 values would")
        print(f"     be fabricated by any imputation strategy (including an 'Unknown'")
        print(f"     category placeholder, which would just become the dominant class).")
        print(f"     The column is dropped rather than kept as a mostly-synthetic feature.")
        df_clean = df_clean.drop(columns=[col])

print("\n--- Missing values remaining after cleaning ---")
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])
print(f"\nShape after cleaning: {df_clean.shape}")


STEP 2: Missing-value handling (percentage-threshold rule)
Rule: <5% missing -> drop rows | 5-30% missing -> impute | >30% -> drop column

deck: 77.22% missing
  -> Over 30% missing: too unreliable to impute confidently.
     Decision: DROP the 'deck' column entirely.
     Justification: at 77.22% missing, more than 3 out of 4 values would
     be fabricated by any imputation strategy (including an 'Unknown'
     category placeholder, which would just become the dominant class).
     The column is dropped rather than kept as a mostly-synthetic feature.

age: 19.87% missing
  -> 5-30%: imputing with median (28.00).

embarked: 0.22% missing
  -> Under 5%: dropping rows with missing embarked (2 row(s) dropped).

embark_town: 0.22% missing
  -> Under 5%: dropping rows with missing embark_town (0 row(s) dropped).

--- Missing values remaining after cleaning ---
Series([], dtype: int64)

Shape after cleaning: (889, 14)


STEP 3: Univariate analysis -- Age and Fare

In [6]:
print("\n" + "=" * 70)
print("STEP 3: Univariate analysis -- age and fare")
print("=" * 70)


for col in ["age", "fare"]:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(df_clean[col], bins=30, color="steelblue", edgecolor="white")
    axes[0].set_title(f"{col} — histogram")
    axes[1].boxplot(df_clean[col], vert=True)
    axes[1].set_title(f"{col} — box plot")
    plt.tight_layout()
    plt.savefig(f"charts/univariate_{col}.png")
    plt.close()
    print(f"Saved charts/univariate_{col}.png")

    q1, q3 = df_clean[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_outliers = ((df_clean[col] < lower) | (df_clean[col] > upper)).sum()
    print(f"{col}: IQR outliers = {n_outliers}  (bounds: [{lower:.2f}, {upper:.2f}])")

fare_mean = df_clean["fare"].mean()
fare_median = df_clean["fare"].median()
fare_mode = df_clean["fare"].mode()[0]
print(f"\nfare -- mean: {fare_mean:.2f}, median: {fare_median:.2f}, mode: {fare_mode:.2f}")

if fare_mean > fare_median > fare_mode:
    skew_text = ("right-skewed (mean > median > mode) -- a long tail of a few very "
                 "expensive fares pulls the mean above the median and mode.")
elif fare_mean < fare_median < fare_mode:
    skew_text = "left-skewed (mean < median < mode)."
else:
    skew_text = "roughly symmetric (mean, median, and mode are close together)."
print(f"fare distribution: {skew_text}")



STEP 3: Univariate analysis -- age and fare


C:\Users\TEMP.DESKTOP-K14BCI3.000\AppData\Local\Temp\ipykernel_17292\707563232.py:10: MatplotlibDeprecationWarning: vert: bool was deprecated in Matplotlib 3.11 and will be removed in 3.13. Use orientation: {'vertical', 'horizontal'} instead.
  axes[1].boxplot(df_clean[col], vert=True)


Saved charts/univariate_age.png
age: IQR outliers = 65  (bounds: [2.50, 54.50])


C:\Users\TEMP.DESKTOP-K14BCI3.000\AppData\Local\Temp\ipykernel_17292\707563232.py:10: MatplotlibDeprecationWarning: vert: bool was deprecated in Matplotlib 3.11 and will be removed in 3.13. Use orientation: {'vertical', 'horizontal'} instead.
  axes[1].boxplot(df_clean[col], vert=True)


Saved charts/univariate_fare.png
fare: IQR outliers = 114  (bounds: [-26.76, 65.66])

fare -- mean: 32.10, median: 14.45, mode: 8.05
fare distribution: right-skewed (mean > median > mode) -- a long tail of a few very expensive fares pulls the mean above the median and mode.


STEP 4: Bivariate analysis -- survival rate breakdowns

In [7]:
print("\n" + "=" * 70)
print("STEP 4: Bivariate analysis -- survival rates (boolean masking)")
print("=" * 70)

print("\n(a) Survival rate by sex:")
for s in sorted(df_clean["sex"].unique()):
    mask = df_clean["sex"] == s
    rate = df_clean.loc[mask, "survived"].mean()
    print(f"  sex={s}: {rate:.2%}  (n={mask.sum()})")

print("\n(b) Survival rate by pclass:")
for p in sorted(df_clean["pclass"].unique()):
    mask = df_clean["pclass"] == p
    rate = df_clean.loc[mask, "survived"].mean()
    print(f"  pclass={p}: {rate:.2%}  (n={mask.sum()})")

print("\n(c) Survival rate by sex AND pclass:")
for s in sorted(df_clean["sex"].unique()):
    for p in sorted(df_clean["pclass"].unique()):
        mask = (df_clean["sex"] == s) & (df_clean["pclass"] == p)
        rate = df_clean.loc[mask, "survived"].mean()
        print(f"  sex={s} & pclass={p}: {rate:.2%}  (n={mask.sum()})")


STEP 4: Bivariate analysis -- survival rates (boolean masking)

(a) Survival rate by sex:
  sex=female: 74.04%  (n=312)
  sex=male: 18.89%  (n=577)

(b) Survival rate by pclass:
  pclass=1: 62.62%  (n=214)
  pclass=2: 47.28%  (n=184)
  pclass=3: 24.24%  (n=491)

(c) Survival rate by sex AND pclass:
  sex=female & pclass=1: 96.74%  (n=92)
  sex=female & pclass=2: 92.11%  (n=76)
  sex=female & pclass=3: 50.00%  (n=144)
  sex=male & pclass=1: 36.89%  (n=122)
  sex=male & pclass=2: 15.74%  (n=108)
  sex=male & pclass=3: 13.54%  (n=347)


STEP 5: Correlation matrix (exactly 6 specified numeric columns)

In [8]:
print("\n" + "=" * 70)
print("STEP 5: Correlation matrix -- survived, pclass, age, sibsp, parch, fare")
print("=" * 70)
print("(adult_male and alone are excluded -- they are derived/redundant flags,")
print("not independently measured features.)")

corr_cols = ["survived", "pclass", "age", "sibsp", "parch", "fare"]
corr_matrix = df_clean[corr_cols].corr()
print("\n", corr_matrix.round(2))

plt.figure(figsize=(7, 6))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Correlation matrix (6 numeric columns)")
plt.tight_layout()
plt.savefig("charts/correlation_heatmap.png")
plt.close()
print("\nSaved charts/correlation_heatmap.png")

pairs = []
for i in range(len(corr_cols)):
    for j in range(i + 1, len(corr_cols)):
        pairs.append((corr_cols[i], corr_cols[j], corr_matrix.iloc[i, j]))
pairs_sorted = sorted(pairs, key=lambda x: abs(x[2]), reverse=True)

print("\nTop 2 strongest correlations (by absolute value):")
for a, b, r in pairs_sorted[:2]:
    direction = "increase" if r > 0 else "decrease"
    print(f"  {a} vs {b}: r = {r:.2f} -- as {a} rises, {b} tends to {direction}.")


STEP 5: Correlation matrix -- survived, pclass, age, sibsp, parch, fare
(adult_male and alone are excluded -- they are derived/redundant flags,
not independently measured features.)

           survived  pclass   age  sibsp  parch  fare
survived      1.00   -0.34 -0.07  -0.03   0.08  0.26
pclass       -0.34    1.00 -0.34   0.08   0.02 -0.55
age          -0.07   -0.34  1.00  -0.23  -0.17  0.09
sibsp        -0.03    0.08 -0.23   1.00   0.41  0.16
parch         0.08    0.02 -0.17   0.41   1.00  0.22
fare          0.26   -0.55  0.09   0.16   0.22  1.00

Saved charts/correlation_heatmap.png

Top 2 strongest correlations (by absolute value):
  pclass vs fare: r = -0.55 -- as pclass rises, fare tends to decrease.
  sibsp vs parch: r = 0.41 -- as sibsp rises, parch tends to increase.


STEP 6: Multivariate data story (4+ charts, each with interpretation)

In [9]:
print("\n" + "=" * 70)
print("STEP 6: Multivariate data story")
print("=" * 70)

# Chart 1 -- survival rate by class and sex
plt.figure(figsize=(7, 5))
sns.barplot(data=df_clean, x="pclass", y="survived", hue="sex")
plt.title("Survival rate by class and sex")
plt.ylabel("Survival rate")
plt.tight_layout()
plt.savefig("charts/story_1_survival_by_class_sex.png")
plt.close()
print("\nChart 1 saved: story_1_survival_by_class_sex.png")
print("Interpretation: Women survived at a dramatically higher rate than men in every")
print("passenger class, and 1st-class passengers of both sexes survived more often than")
print("2nd or 3rd class -- consistent with 'women and children first' evacuation norms")
print("combined with unequal lifeboat access by class.")

# Chart 2 -- age distribution by survival outcome
plt.figure(figsize=(7, 5))
sns.boxplot(data=df_clean, x="survived", y="age")
plt.title("Age distribution by survival outcome")
plt.tight_layout()
plt.savefig("charts/story_2_age_by_survived.png")
plt.close()
print("\nChart 2 saved: story_2_age_by_survived.png")
print("Interpretation: Survivors skew slightly younger, with a lower median age than")
print("non-survivors, hinting at children being prioritized -- but the overlap between")
print("the two boxes is large, so age alone is a weak predictor on its own.")

# Chart 3 -- fare vs age scatter, colored by survival
plt.figure(figsize=(7, 5))
sns.scatterplot(data=df_clean, x="age", y="fare", hue="survived", alpha=0.6)
plt.title("Fare vs Age, colored by survival")
plt.tight_layout()
plt.savefig("charts/story_3_fare_age_scatter.png")
plt.close()
print("\nChart 3 saved: story_3_fare_age_scatter.png")
print("Interpretation: Survivors cluster more heavily in the higher-fare band across all")
print("ages, while non-survivors are denser in the low-fare region -- reinforcing that")
print("ticket price (a proxy for class/cabin location) mattered more to survival odds")
print("than age by itself.")

# Chart 4 -- survival rate by embarkation port
plt.figure(figsize=(7, 5))
sns.barplot(data=df_clean, x="embarked", y="survived")
plt.title("Survival rate by port of embarkation")
plt.ylabel("Survival rate")
plt.tight_layout()
plt.savefig("charts/story_4_survival_by_embarked.png")
plt.close()
print("\nChart 4 saved: story_4_survival_by_embarked.png")
print("Interpretation: Passengers who boarded at Cherbourg ('C') show a noticeably")
print("higher survival rate than those from Southampton ('S') or Queenstown ('Q') --")
print("likely because Cherbourg passengers were disproportionately 1st-class, so this is")
print("probably a proxy for class rather than an independent boarding-port effect.")


STEP 6: Multivariate data story

Chart 1 saved: story_1_survival_by_class_sex.png
Interpretation: Women survived at a dramatically higher rate than men in every
passenger class, and 1st-class passengers of both sexes survived more often than
2nd or 3rd class -- consistent with 'women and children first' evacuation norms
combined with unequal lifeboat access by class.

Chart 2 saved: story_2_age_by_survived.png
Interpretation: Survivors skew slightly younger, with a lower median age than
non-survivors, hinting at children being prioritized -- but the overlap between
the two boxes is large, so age alone is a weak predictor on its own.

Chart 3 saved: story_3_fare_age_scatter.png
Interpretation: Survivors cluster more heavily in the higher-fare band across all
ages, while non-survivors are denser in the low-fare region -- reinforcing that
ticket price (a proxy for class/cabin location) mattered more to survival odds
than age by itself.

Chart 4 saved: story_4_survival_by_embarked.png
Int

STEP 7: Exploratory z-score standardization check (age, fare)

In [10]:
print("\n" + "=" * 70)
print("STEP 7: Exploratory z-score standardization check (age, fare)")
print("=" * 70)
print("(EDA-stage sanity check only -- 02_modeling.py performs its own train-only")
print("scaling and does not reuse these columns.)")

print("\nBefore standardization:")
print(df_clean[["age", "fare"]].agg(["mean", "std"]).round(3))

df_check = df_clean.copy()
for col in ["age", "fare"]:
    df_check[col + "_z"] = (df_check[col] - df_check[col].mean()) / df_check[col].std()

print("\nAfter standardization (z-score):")
print(df_check[["age_z", "fare_z"]].agg(["mean", "std"]).round(4))
print("\nMeans are ~0 and standard deviations are ~1, confirming the transform worked.")

print("\n" + "=" * 70)
print("01_eda.py complete. titanic.csv and charts/ are ready for 02_modeling.py.")
print("=" * 70)


STEP 7: Exploratory z-score standardization check (age, fare)
(EDA-stage sanity check only -- 02_modeling.py performs its own train-only
scaling and does not reuse these columns.)

Before standardization:
         age    fare
mean  29.315  32.097
std   12.985  49.698

After standardization (z-score):
      age_z  fare_z
mean    0.0     0.0
std     1.0     1.0

Means are ~0 and standard deviations are ~1, confirming the transform worked.

01_eda.py complete. titanic.csv and charts/ are ready for 02_modeling.py.
